In [1]:
import datetime
import os

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
sns.set_theme(style="whitegrid", palette="deep")
task_palette = sns.color_palette("deep")
plt.rcParams["text.usetex"] = True
plt.rcParams.update({"figure.titlesize": "small"})
plt.rcParams.update({"axes.titlesize": "small"})
plt.rcParams.update({"axes.labelsize": "small"})
plt.rcParams.update({"ytick.labelsize": "small"})
plt.rcParams.update({"xtick.labelsize": "small"})
plt.rcParams.update({"legend.fontsize": "small"})

In [ ]:
path = "./results/results-mcs_scale_bfs-2026-04-15_20-29-06.csv"
df = pd.read_csv(path, sep=";")
df

In [ ]:
df["duration_s"] = df["duration_ns"] / 1e9

In [ ]:
df["search_type"] = df["use_idlesim"].apply(lambda x: "ACBFS" if x else "BFS")

In [ ]:
df = df.dropna(how="any")

In [ ]:
s = df.shape[0] * [True]
df_plot = df.loc[s]
df_piv = df_plot.copy().pivot(
    index="taskset_position",
    columns="search_type",
    values=["duration_s", "visited_count", "max_period", "nbt"],
)
df_piv.columns = list(map(lambda x: "_".join(x), df_piv.columns))

In [ ]:
df_piv = df_piv.dropna(how="any")

In [ ]:
df_piv["Number of tasks"] = df_piv["nbt_BFS"].astype(int)
df_piv["Period max"] = df_piv["max_period_ACBFS"].astype(int)
df_piv["n"] = df_piv["nbt_BFS"].astype(int)
df_piv["T max"] = df_piv["max_period_ACBFS"].astype(int)

In [ ]:
f, ax = plt.subplots(figsize=(6, 3))

sns.scatterplot(
    data=df_piv,
    x="duration_s_BFS",
    y="duration_s_ACBFS",
    style="T max",
    hue="n",
    ax=ax,
    palette=task_palette,
)
ax.set(xscale="log", yscale="log")
# ax.axis("equal")

lims = [
    10**-5.1,
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, "k--", alpha=0.75, zorder=1)

ax.set(xlim=lims, ylim=lims)

ax.set_xlabel("BFS")
ax.set_ylabel("ACBFS")
ax.set_title("Execution time (s)")

# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.1))

f.suptitle(r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$", y=1.02)

h, l = ax.get_legend_handles_labels()
ax.legend_.remove()
leg = [
    "$n$",
    "2",
    "3",
    "4",
    "5",
    "6",
    "$T^{\mathsf{max}}$",
    "10",
    "15",
    "20",
    "25",
    "30",
]
f.legend(h, leg, ncol=2, loc="upper left", bbox_to_anchor=(0.15, 0.85), framealpha=1)


x = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_BFS"])
y = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_ACBFS"])
m, b = np.polyfit(x, y, 1)
x_reg = np.linspace(lims, 100)
x_reg = np.log10(x_reg)
plt.plot(10**x_reg, 10 ** (m * x_reg + b), color="k", ls=":")

In [ ]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_bfs_acbfs.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/bfs_acbfs.pdf",
    bbox_inches="tight",
)

In [ ]:
path = "./results/results-mcs_scale01-2026-04-15_20-29-48.csv"

df_sp = pd.read_csv(path, sep=";")
df_sp

In [ ]:
df_sp["taskset_file"].value_counts()

In [ ]:
df_sp.groupby("taskset_file")["taskset_position"].max()

In [ ]:
df_sp["duration_s"] = df_sp["duration_ns"] / 1e9

In [ ]:
df_sp["timeout_int"] = df_sp["timeout"].astype(int)
df_sp.groupby(["nbt"])["timeout_int"].agg(["count", "sum"])

In [ ]:
df_sp.groupby("nbt")["duration_s"].describe()

In [ ]:
# f, axs = plt.subplots(2, 1, figsize=(3.5*52/98*2, 3.5), gridspec_kw={"hspace": 0.25})
f, axs = plt.subplots(
    2, 1, figsize=(3.2 / 1.05, 3.2), gridspec_kw={"hspace": 0.25}, sharey=True
)


ax1 = axs[0]
ax2 = ax1.twinx()

s = df_sp["taskset_file"].str.contains("task")
dimension = "nbt"

# s = df_sp["taskset_file"].str.contains("period")
# dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


n_xp = 50

x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Number of tasks ($n$), $T^{\mathsf{max}} = 30$")
ax1.set_ylabel("Execution time (s)")
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()

ax1.xaxis.tick_top()
ax1.xaxis.set_label_position("top")
ax1.xaxis.set_ticks_position("none")
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")
ax1.tick_params(axis="x", which="major", pad=0)


ax1 = axs[1]
ax2 = ax1.twinx()

# s = df_sp["taskset_file"].str.contains("task")
# dimension = "nbt"

s = df_sp["taskset_file"].str.contains("period")
dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Period max ($T^{\mathsf{max}}$), $n = 5$")
ax1.set_ylabel("Execution time (s)", labelpad=10)
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")

f.suptitle(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$",
    # x=0.55,
    # y=0.96,
    y=1.05,
)

blue_line = mlines.Line2D(
    [],
    [],
    color=task_palette[0],
    marker="o",
    linestyle="-",
    markeredgecolor="w",
    label="Execution time (s)",
    ms=9,
)

orange_box = mpatches.Patch(
    hatch="//", label="Timeouts", edgecolor=task_palette[1], facecolor="white"
)

new_handles = [blue_line, orange_box]


ax1.legend(
    handles=new_handles,
    loc="upper center",
    ncol=2,
    framealpha=0,
    columnspacing=0.8,
    bbox_to_anchor=(0.5, 1.28),
)

ax1.get_ylim()


f.tight_layout()

In [ ]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_tmax_ntask.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/tmax_ntask.pdf",
    bbox_inches="tight",
)

## Schedulability and Time plots for sched test suite

In [3]:
path = "./results/results-mcs_schedulability-2026-05-13_05-36-27.csv"
df_sch = pd.read_csv(path, sep=";")

df_sch

,safe_oracles,unsafe_oracles,use_case,scheduler,search_algorithms,taskset_file,taskset_position,U,Uv,nbt,...,probability_of_HI,min_period,max_period,timeout,is_safe_0,automaton_depth_0,visited_count_0,duration_ns_0,is_safe,duration_ns
0,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,6,0.50,0.496329,5.0,...,0.5,5.0,30.0,False,True,17,1857,7072312,True,7072312
1,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,3,0.50,0.499537,5.0,...,0.5,5.0,30.0,False,True,9,1806,12335853,True,12335853
2,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,4,0.50,0.497891,5.0,...,0.5,5.0,30.0,False,True,21,8424,35236078,True,35236078
3,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,2,0.50,0.497348,5.0,...,0.5,5.0,30.0,False,True,23,8483,34937538,True,34937538
4,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,12,0.50,0.498810,5.0,...,0.5,5.0,30.0,False,True,17,4251,15254134,True,15254134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32705,[],['hi-over-demand'],EDF-VD (PDFS),edfvd,['pdfs'],outputs/20251022_055741-scheduling-rtss.txt,9627,0.95,0.954015,5.0,...,0.5,5.0,30.0,False,True,209209,1699412,9391859977,True,9391859977
32706,[],['hi-over-demand'],EDF-VD (PDFS),edfvd,['pdfs'],outputs/20251022_055741-scheduling-rtss.txt,10849,1.00,0.995568,5.0,...,0.5,5.0,30.0,False,False,1950988,1951149,5083850343,False,5083850343
32707,[],['hi-over-demand'],EDF-VD (PDFS),edfvd,['pdfs'],outputs/20251022_055741-scheduling-rtss.txt,10721,1.00,0.998066,5.0,...,0.5,5.0,30.0,False,False,2521281,2521759,7455282484,False,7455282484
32708,[],['hi-over-demand'],EDF-VD (PDFS),edfvd,['pdfs'],outputs/20251022_055741-scheduling-rtss.txt,10572,1.00,0.995149,5.0,...,0.5,5.0,30.0,False,False,4687199,4806225,13337719698,False,13337719698


In [4]:
# Patch: add more schedulability results
patch_paths = ["./results/results-mcs_schedulability-2026-05-13_14-58-50.csv"]
existing_use_cases = set(df_sch["use_case"].unique())
for patch_path in patch_paths:
    df_patch = pd.read_csv(patch_path, sep=";")
    df_patch = df_patch[~df_patch["use_case"].isin(existing_use_cases)]
    df_sch = pd.concat([df_sch, df_patch], ignore_index=True)

df_sch

,safe_oracles,unsafe_oracles,use_case,scheduler,search_algorithms,taskset_file,taskset_position,U,Uv,nbt,...,probability_of_HI,min_period,max_period,timeout,is_safe_0,automaton_depth_0,visited_count_0,duration_ns_0,is_safe,duration_ns
0,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,6,0.50,0.496329,5.0,...,0.5,5.0,30.0,False,True,17,1857,7072312,True,7072312
1,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,3,0.50,0.499537,5.0,...,0.5,5.0,30.0,False,True,9,1806,12335853,True,12335853
2,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,4,0.50,0.497891,5.0,...,0.5,5.0,30.0,False,True,21,8424,35236078,True,35236078
3,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,2,0.50,0.497348,5.0,...,0.5,5.0,30.0,False,True,23,8483,34937538,True,34937538
4,[],['hi-over-demand'],EDF-VD (AC),edfvd,['acbfs'],outputs/20251022_055741-scheduling-rtss.txt,12,0.50,0.498810,5.0,...,0.5,5.0,30.0,False,True,17,4251,15254134,True,15254134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43705,[],['hi-over-demand'],EDF-VD (P),edfvd,['pbfs'],outputs/20251022_055741-scheduling-rtss.txt,9654,0.95,0.946787,5.0,...,0.5,5.0,30.0,False,True,1045044,12191898,19331890147,True,19331890147
43706,[],['hi-over-demand'],EDF-VD (P),edfvd,['pbfs'],outputs/20251022_055741-scheduling-rtss.txt,9698,0.95,0.946047,5.0,...,0.5,5.0,30.0,False,True,1863540,13469760,19772008953,True,19772008953
43707,[],['hi-over-demand'],EDF-VD (P),edfvd,['pbfs'],outputs/20251022_055741-scheduling-rtss.txt,9200,0.95,0.946706,5.0,...,0.5,5.0,30.0,False,True,2034900,24348789,34356114450,True,34356114450
43708,[],['hi-over-demand'],EDF-VD (P),edfvd,['pbfs'],outputs/20251022_055741-scheduling-rtss.txt,9813,0.95,0.949090,5.0,...,0.5,5.0,30.0,False,True,2960958,23766499,31119655386,True,31119655386


Taskset Parameters:
- safe_oracles: safe oracles applied to the search algorithm
- unsafe_oracles: unsafe oracles applied to the search algorithm
- use_case: name of the scheduler and chained search algorithms configuration
- taskset_position: position of the taskset in the original generation

Split the raw table into:
- df_edfvd_test: from EDFVD_test column of the raw table, rename the use_case into "EDF-VD (sufficient)"
- df_stages[i]: from stage i of search algorithm of the raw table.
    - stage: i
    - search_algorithm: from search_algorithms[i]
    - schedulable: from is_safe_{i}
    - automaton_depth: from automaton_depth_{i}
    - visited_count: from visited_count_{i}
    - duration_ns: from duration_ns_{i}
- df_combined: for the combined final result of the search algorithm combinations.

In [5]:
# EDF-VD test
df_edfvd_test = df_sch[["taskset_position", "EDFVD_test", "U"]]
df_edfvd_test = df_edfvd_test.rename(columns={"EDFVD_test": "EDF-VD (sufficient)"})
df_edfvd_test["EDF-VD (sufficient)"] = df_edfvd_test["EDF-VD (sufficient)"].astype(bool)
df_edfvd_test = df_edfvd_test.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)
df_edfvd_test["stage"] = -1

df_edfvd_test

,taskset_position,U,use_case,schedulable,stage
0,6,0.50,EDF-VD (sufficient),True,-1
1,3,0.50,EDF-VD (sufficient),True,-1
2,4,0.50,EDF-VD (sufficient),True,-1
3,2,0.50,EDF-VD (sufficient),True,-1
4,12,0.50,EDF-VD (sufficient),True,-1
...,...,...,...,...,...
43705,9654,0.95,EDF-VD (sufficient),False,-1
43706,9698,0.95,EDF-VD (sufficient),False,-1
43707,9200,0.95,EDF-VD (sufficient),False,-1
43708,9813,0.95,EDF-VD (sufficient),False,-1


In [ ]:
# Stages
n_stages = df_sch["search_algorithms"][0].count(",") + 1
df_stages = []

for i in range(n_stages):
    df_stage = df_sch[["taskset_position", "use_case", "U", f"is_safe_{i}", f"automaton_depth_{i}", f"visited_count_{i}", f"duration_ns_{i}"]]
    df_stage = df_stage.rename(columns={f"is_safe_{i}": "schedulable", f"automaton_depth_{i}": "automaton_depth", f"visited_count_{i}": "visited_count", f"duration_ns_{i}": "duration_ns"})
    df_stage["schedulable"] = df_stage["schedulable"].astype(bool)
    df_stage["stage"] = i
    df_stage["search_algorithm"] = df_sch["search_algorithms"].apply(lambda x: x.strip("[]").split(", ")[i].strip("'"))
    df_stages.append(df_stage)

df_stages[-1]

In [6]:
df_total = df_sch[["taskset_position", "use_case", "U", "is_safe", "duration_ns"]]
df_total = df_total.rename(columns={"is_safe": "schedulable"})
df_total["schedulable"] = df_total["schedulable"].astype(bool)
df_total["stage"] = -1

df_total

,taskset_position,use_case,U,schedulable,duration_ns,stage
0,6,EDF-VD (AC),0.50,True,7072312,-1
1,3,EDF-VD (AC),0.50,True,12335853,-1
2,4,EDF-VD (AC),0.50,True,35236078,-1
3,2,EDF-VD (AC),0.50,True,34937538,-1
4,12,EDF-VD (AC),0.50,True,15254134,-1
...,...,...,...,...,...,...
43705,9654,EDF-VD (P),0.95,True,19331890147,-1
43706,9698,EDF-VD (P),0.95,True,19772008953,-1
43707,9200,EDF-VD (P),0.95,True,34356114450,-1
43708,9813,EDF-VD (P),0.95,True,31119655386,-1


In [7]:
df_sched_comb = pd.concat([df_edfvd_test, df_total], ignore_index=True)
df_sched_comb

,taskset_position,U,use_case,schedulable,stage,duration_ns
0,6,0.50,EDF-VD (sufficient),True,-1,NaN
1,3,0.50,EDF-VD (sufficient),True,-1,NaN
2,4,0.50,EDF-VD (sufficient),True,-1,NaN
3,2,0.50,EDF-VD (sufficient),True,-1,NaN
4,12,0.50,EDF-VD (sufficient),True,-1,NaN
...,...,...,...,...,...,...
87415,9654,0.95,EDF-VD (P),True,-1,1.933189e+10
87416,9698,0.95,EDF-VD (P),True,-1,1.977201e+10
87417,9200,0.95,EDF-VD (P),True,-1,3.435611e+10
87418,9813,0.95,EDF-VD (P),True,-1,3.111966e+10


In [8]:
fig_size = (6, 4)

In [15]:
f, ax = plt.subplots(figsize=fig_size)

col_order = df_sched_comb["use_case"].unique()

sns.lineplot(
    data=df_sched_comb,
    x="U",
    y="schedulable",
    hue="use_case",
    ax=ax,
    markers=True,
    errorbar=None,
    style="use_case",
    ms=9,
    hue_order=col_order,
    style_order=col_order,
)

ax.set_ylabel("Schedulability ratio")
ax.set_xlabel("Average utilisation ($U^*$)")

# f.suptitle(
#     r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
#     x=0.55,
#     # y=0.7,
# )

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
)

# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=handles, labels=labels, framealpha=1, loc="lower left")
# ax.legend(
#     handles=handles,
#     labels=labels,
#     loc="upper center",
#     ncol=2,
#     framealpha=0,
#     columnspacing=0.8,
#     bbox_to_anchor=(0.5, 1.4),
# )
sns.move_legend(
    ax,
    loc="center left",
    # ncol=2,
    framealpha=1,
    # columnspacing=0.8,
    # bbox_to_anchor=(0.5, 1.8),
    title="",
)

f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

In [16]:
f.show()

In [11]:
date = datetime.datetime.now().strftime("%Y%m%d")
os.makedirs(f"./figs/{date}", exist_ok=True)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.png",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/scheduling.pdf",
    bbox_inches="tight",
)

Plots the time taken vs average utilization for each algorithm

In [12]:
f, ax = plt.subplots(figsize=fig_size)

df_time = (
    df_sch.loc[:, ["U", "duration_ns", "use_case"]]
    .dropna(how="any")
    .assign(duration_s=lambda d: d["duration_ns"] / 1e9)
    .sort_values(["use_case", "U"])
)

sns.lineplot(
    data=df_time,
    x="U",
    y="duration_s",
    hue="use_case",
    style="use_case",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Time taken (s)")
ax.set_xlabel("Average utilisation ($U^*$)")

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, n=5$",
)


/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

Text(0.5, 1.0, '$P_{\\mathsf{HI}} = 0.5, n=5$')

In [13]:
date = datetime.datetime.now().strftime("%Y%m%d")
os.makedirs(f"./figs/{date}", exist_ok=True)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.png",
    bbox_inches="tight",
)
f.savefig(
    "./figs/edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)

Plot schedulability by stage for a specific algorithm

In [14]:
def plot_schedulability_by_stage(use_case):
    f, ax = plt.subplots(figsize=fig_size)

    df_scheds = []

    df_sched = df_sched_comb.loc[df_sched_comb["use_case"] == use_case]
    df_sched["search_algorithm"] = "Overall"
    df_scheds.append(df_sched)

    for i in range(n_stages):
        df_scheds.append(
            df_stages[i].loc[:, ["U", "schedulable", "use_case", "search_algorithm", "stage"]][df_stages[i]["use_case"] == use_case]
        )

    df_combined = pd.concat(df_scheds, ignore_index=True)
    search_algorithms = df_combined["search_algorithm"].unique()

    sns.lineplot(
        data=df_combined,
        x="U",
        y="schedulable",
        ax=ax,
        markers=True,
        errorbar=None,
        ms=9,
        hue="search_algorithm",
        hue_order=search_algorithms,
        style="search_algorithm",
        style_order=search_algorithms,
    )

    ax.set_ylabel("Schedulability ratio")
    ax.set_xlabel("Average utilisation ($U^*$)")
    ax.set_title(f'Schedulability by stages - {use_case}')

    return f

In [ ]:
use_cases = ["EDF-VD (PAC-AC)", "EDF-VD (PAC-P-AC)", "LWLF (PAC-AC)", "LWLF (PAC-P-AC)"]

In [ ]:
for use_case in use_cases:
    f = plot_schedulability_by_stage(use_case)
    date = datetime.datetime.now().strftime('%Y%m%d')
    os.makedirs(f"./figs/{date}", exist_ok=True)
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_schedulability_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_schedulability_by_stage_{use_case.replace(' ', '_')}.png",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/schedulability_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )

Plots the time taken by stage for a specific algorithm

In [ ]:
def plot_time_by_stage(use_case):
    f, ax = plt.subplots(figsize=fig_size)

    df_times = []

    df_time = (
        df_sch.loc[:, ["U", "duration_ns", "use_case"]][df_sch["use_case"] == use_case]
        .dropna(how="any")
        .assign(duration_s=lambda d: d["duration_ns"] / 1e9)
        .sort_values(["U"])
    )
    df_time["search_algorithm"] = "Total"
    df_times.append(df_time)

    for i in range(n_stages):
        df_times.append(
            df_stages[i].loc[:, ["U", f"duration_ns", "use_case", "search_algorithm", "stage"]][df_sch["use_case"] == use_case]
            .dropna(how="any")
            .assign(duration_s=lambda d: d[f"duration_ns"] / 1e9)
            .sort_values(["U"])
        )

    df_combined = pd.concat(df_times, ignore_index=True)

    sns.lineplot(
        data=df_combined,
        x="U",
        y="duration_s",
        hue="search_algorithm",
        style="search_algorithm",
        marker="x",
        ax=ax,
    )

    ax.set_yscale("log")
    ax.set_ylabel("Time taken (s)")
    ax.set_xlabel("Average utilisation ($U^*$)")
    ax.set_title(f'Time by stages - {use_case}')

    return f

In [ ]:
for use_case in use_cases:
    f = plot_time_by_stage(use_case)
    date = datetime.datetime.now().strftime('%Y%m%d')
    os.makedirs(f"./figs/{date}", exist_ok=True)
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_by_stage_{use_case.replace(' ', '_')}.png",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/time_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )